# MarketMind AI — Verified Synthetic Market Data Analysis

This updated notebook demonstrates **deterministic** Python research tools, in contrast to the separate **optional LLM tool-calling** agent. Inputs are synthetic (seed 42). None of the quoted metrics is a real-market measurement, trading signal, causal result or investment return.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.sample_data_generator import generate_synthetic_market_data
from src.data_loader import validate_market_data
from src.feature_engineering import add_bid_ask_imbalance, merge_trade_snapshot, add_price_features
from src.resampling import downsample_market_data
from src.order_book_analysis import classify_last_price, category_proportions, detect_free_hole_additions
from src.visualization import plot_cumulative_return_liquidity
from tools.market_tools import MarketToolbox, AnalysisScope
from agents.workflow import run_offline

## 1. Generate synthetic snapshots and trades

Order book snapshots update every 3 seconds; synthetic trade records update every 15 seconds. No private raw CSV files are used.

In [ ]:
snap, trades = generate_synthetic_market_data(n_stocks=3, dates=[20210104], n_snapshots=1200, seed=42)
print("Snapshot / trade rows:", len(snap), len(trades))
print("Quality snapshots:", validate_market_data(snap, "snapshot")["passed"])
print("Quality trades:", validate_market_data(trades, "trade")["passed"])
snap.head(3)

## 2. Quote spread, best-level imbalance and Top-5 OBI

All statistics are computed with pandas/NumPy, not inferred by an LLM. They measure displayed quotes/depth, not causal order-flow impact.

In [ ]:
features = add_bid_ask_imbalance(snap)
features[["spread", "relative_spread", "bid_ask_imba", "obi5", "bid_depth_5", "ask_depth_5"]].describe().round(5)

## 3. Causally safe trade/snapshot alignment

Backward-only as-of matching never uses a trade with a timestamp later than its snapshot. Stale observations beyond 60 seconds are left missing.

In [ ]:
merged = add_price_features(merge_trade_snapshot(features, trades, max_staleness_seconds=60))
assert (merged["trade_datetime"].dropna() <= merged.loc[merged["trade_datetime"].notna(), "datetime"]).all()
print("Aligned rows:", len(merged), "Exact observed trade rows:", int(merged.trade_observed.sum()))
merged[["datetime", "trade_datetime", "trade_observed", "trade_age_seconds", "last"]].head(7)

## 4. Resampling and interpretability

`bid1p_realized_quote_vol` is a within-bin quote-price volatility proxy, **not** annualized realized return volatility.

In [ ]:
down = downsample_market_data(merged, interval_minutes=10)
print("Buckets total:", len(down))
down[["id", "datetime", "bid1p", "maxBid1p", "timesBid1pChanged", "bid1p_realized_quote_vol"]].head(6)

## 5. Exact observed trades and new visible-level *candidates*

Last-price category is only computed where a trade was actually observed. Candidate events are transitions in the five displayed price levels, **not** individual order messages.

In [ ]:
classified=classify_last_price(merged,observed_only=True)
print("Classified true trade observations:", int(classified.last_category.notna().sum()))
print("Non-observed snapshots classified:", int(classified.loc[~classified.trade_observed, "last_category"].notna().sum()))
print("Candidate newly visible price levels across demo:", len(detect_free_hole_additions(snap)))
category_proportions(classified).round(3)

## 6. Research workflow and evidence trace

**Mode below is offline deterministic**, not a real model selecting tools. For actual LLM function-calling, use CLI `python run_demo.py --mode llm` with a configured API key.

In [ ]:
scope = AnalysisScope(stock_id=1, trading_date=20210104, start_time="09:40", end_time="10:30", interval_minutes=10)
toolbox=MarketToolbox(output_dir=ROOT/"reports"/"generated")
run=run_offline(scope,toolbox)
print(run.answer[:1600]); print("Executed tools:", [t["tool"] for t in run.trace])

## Summary / next evidence gates

This notebook demonstrates quote calculations, typed read-only tools, data scope, and an auditable deterministic research workflow. For an AI Agent product case, the optional tool-calling mode must be tested on **real model calls**, validated for hallucinations and compared with researcher-task baselines before any model accuracy/time savings claims are made.